In [1]:
from pathlib import Path
import sys
import traceback
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import data_analysis.mea_analysis as mea
import data_analysis.loading_and_saving as las
import data_analysis.plotting as dap

In [2]:
BRW_BASE_FOLDER = Path("/media/ferdinand-forberger/RawDataBackUp")
MAIN_FOLDER = Path("/media/ferdinand-forberger/Seagate Portable Drive")
EVENTS_DURATIONS_PATH = MAIN_FOLDER / "events_durations.xlsx"
GRAPH_METRICS_PATH = MAIN_FOLDER / "graph_metrics.xlsx"
OVERWRITE = False

In [3]:
df_overview = mea.get_overview(MAIN_FOLDER)
df_overview = df_overview[df_overview["sles_saved"]==True]
df_overview.reset_index(inplace=True, drop=True)
df_overview["image_path"]= df_overview["folder_path"].astype(str) + "/org_image.png"
df_overview["mask_path"] = df_overview["folder_path"].astype(str) + "/mask.png"

In [4]:
colors = dap.get_c_map_colors()

In [5]:
df_metadata = pd.read_excel("/media/ferdinand-forberger/RawDataBackUp/meta_data_file.xlsx")
df_metadata["folder"] = df_metadata["animal_number"].astype(str) + "_" + df_metadata["slice_number"].astype(str)
df_overview = pd.merge(df_overview, df_metadata, left_on="folder", right_on="folder", how="left")

In [6]:
durs = []
rates = []
for i, row in df_overview.iterrows():
    recording = las.get_rec_obj(row["bin_path_x"])
    recording_dur = recording.get_duration()
    sle = np.load(row["sle_path"])
    durs.append(np.mean(sle[1,:] - sle[0,:])/mea.SAMPLING_RATE)
    rates.append(sle.shape[1]/recording_dur)
df_overview["sle_dur"] = durs
df_overview["sle_rate"] = rates

In [7]:
df_overview = df_overview.sort_values(by="group", ascending=True)
df_overview.reset_index(inplace=True, drop=True)
df_overview.to_excel(EVENTS_DURATIONS_PATH)

In [15]:
df_graph_metrics = pd.read_excel(GRAPH_METRICS_PATH)
print(df_graph_metrics.columns)
df_graph_metrics = df_graph_metrics[["folder","global_efficiency","average_node_strength","average_clustering","modularity_score"]]

Index(['folder', 'folder_path', 'bin_path_x', 'analyzed', 'sle_path',
       'sles_saved', 'image_path', 'mask_path', 'animal_number',
       'slice_number', 'analyzed_bool', 'original_recording_base_path',
       'target_folder_base_path', 'start_time', 'end_time',
       'kilosort_output_path', 'bin_path_y', 'brw_path', 'group', 'preincub',
       'durations', 'slcc_path', 'slcc_path_SLES', 'sttc_path_no_SLE',
       'global_efficiency', 'average_clustering', 'average_node_strength',
       'modularity_score'],
      dtype='object')


In [16]:
df_data = df_overview[["folder", "group", "sle_dur"]]
df_data = pd.merge(df_data, df_graph_metrics, on="folder")

In [17]:
df_data

,folder,group,sle_dur,global_efficiency,average_node_strength,average_clustering,modularity_score
0,8606_6,Sham,0.910935,1.137366,2.217343,0.186118,1.954519e-01
1,8606_2,Sham,0.842579,1.168116,11.299825,0.199619,5.454927e-02
2,8562_5,Sham,0.893065,1.188207,5.626489,0.244167,5.653013e-02
3,8562_7,Sham,0.924763,1.080322,28.201682,0.159289,1.004490e-01
4,8606_1,Sham,0.451083,1.104196,0.357180,0.306655,0.000000e+00
5,8605_6,Sham,1.007438,1.120005,8.595738,0.164211,4.963998e-02
6,8605_5,Sham,0.498982,1.153267,13.682102,0.196627,5.426280e-02
7,8605_2,Sham,0.848031,1.201948,3.699251,0.311142,2.924549e-03
8,8605_1,Sham,0.531062,1.146371,4.991324,0.203815,1.341795e-01
9,8606_5,Sham,0.817976,1.153399,5.280076,0.190533,4.736233e-02


In [18]:
import seaborn as sns
from scipy.stats import spearmanr

In [19]:
df_data

,folder,group,sle_dur,global_efficiency,average_node_strength,average_clustering,modularity_score
0,8606_6,Sham,0.910935,1.137366,2.217343,0.186118,1.954519e-01
1,8606_2,Sham,0.842579,1.168116,11.299825,0.199619,5.454927e-02
2,8562_5,Sham,0.893065,1.188207,5.626489,0.244167,5.653013e-02
3,8562_7,Sham,0.924763,1.080322,28.201682,0.159289,1.004490e-01
4,8606_1,Sham,0.451083,1.104196,0.357180,0.306655,0.000000e+00
5,8605_6,Sham,1.007438,1.120005,8.595738,0.164211,4.963998e-02
6,8605_5,Sham,0.498982,1.153267,13.682102,0.196627,5.426280e-02
7,8605_2,Sham,0.848031,1.201948,3.699251,0.311142,2.924549e-03
8,8605_1,Sham,0.531062,1.146371,4.991324,0.203815,1.341795e-01
9,8606_5,Sham,0.817976,1.153399,5.280076,0.190533,4.736233e-02


In [21]:
for group in df_data['group'].unique():
    
    for metric in ["global_efficiency","average_node_strength","average_clustering","modularity_score"]:
        subset = df_data[df_data['group'] == group]
        corr, p_value = spearmanr(subset['sle_dur'], subset[metric])
        print(f"Spearman correlation between NB duration and {metric} for group {group}: {corr:.3f} (p-value: {p_value:.3f})")
    

Spearman correlation between NB duration and global_efficiency for group Sham: -0.139 (p-value: 0.701)
Spearman correlation between NB duration and average_node_strength for group Sham: 0.297 (p-value: 0.405)
Spearman correlation between NB duration and average_clustering for group Sham: -0.576 (p-value: 0.082)
Spearman correlation between NB duration and modularity_score for group Sham: 0.370 (p-value: 0.293)
Spearman correlation between NB duration and global_efficiency for group Tumor: 0.552 (p-value: 0.063)
Spearman correlation between NB duration and average_node_strength for group Tumor: 0.329 (p-value: 0.297)
Spearman correlation between NB duration and average_clustering for group Tumor: 0.287 (p-value: 0.366)
Spearman correlation between NB duration and modularity_score for group Tumor: -0.343 (p-value: 0.276)
